# Usage | 5. Custom data imports
This notebook explains how custom data sets can be imported into growbikenet to adapt to a specific situation, growing more realistically.

**Parameters covered**: `import_files`

We start every Usage notebook with the standard way of importing growbikenet:

In [ ]:
import growbikenet as gbn

## Save and import street network

Here we work with Athens. When running growbikenet, it uses OSMnx to fetch street network data from [OSM](https://www.openstreetmap.org/). OSMnx has data caching implemented, so when running it multiple times on the same city, it will load the cached json data. Unfortunately, loading json is slow. To speed up this process, we can first download and save the (undirected) street network locally, under `Barcelona_street_network.gpkg`, so we can load that file fast into growbikenet whenever we will use it (because gpkg files are loaded fast):

In [ ]:
import osmnx as ox
g = ox.graph_from_place("Municipality of Athens", network_type='drive')
ox.io.save_graph_geopackage(g, "Athens_street_network.gpkg")

Now, we can use this local street network file via the `import_files` parameter:

In [ ]:
edges_ranked = gbn.growbikenet("Municipality of Athens",
                               import_files={"street_network":"Athens_street_network.gpkg"},)

This way, the street network file can also come from any other data provider than OSM, given that the data is in the same format.

## Save and import bike network

Anaologously, a bike network can be downloaded, saved, and imported. It only needs some preparation to set up a [custom filter](https://github.com/gboeing/osmnx-examples/blob/main/notebooks/08-custom-filters-infrastructure.ipynb):

In [ ]:
custom_filter = ['["cycleway"~"track"]',
          '["highway"~"cycleway"]',
          '["highway"~"path"]["bicycle"~"designated"]',
          '["cycleway:right"~"track"]',
          '["cycleway:left"~"track"]',
          '["cycleway:both"~"track"]',
          '["cyclestreet"]',
          '["highway"~"living_street"]'
        ]
for custom_tag in ["cycleway", "bicycle", "cycleway:right", "cycleway:left", "cycleway:both", "cyclestreet"]:
    if custom_tag not in ox.settings.useful_tags_way:
        ox.settings.useful_tags_way.extend(custom_tag)

In [ ]:
g = ox.graph_from_place("Municipality of Athens",
                        custom_filter=custom_filter,
                        retain_all=True,) #fetch all connected components
ox.io.save_graph_geopackage(g, "Athens_bike_network.gpkg")

In [ ]:
edges_ranked = gbn.growbikenet("Municipality of Athens",
                               existing_network_spacing='auto',
                               seed_point_linking ='triangulate_delaunay',
                               import_files={"street_network":"Athens_street_network.gpkg",
                                             "bike_network":"Athens_bike_network.gpkg"},)

In [ ]:
import folium
viz = edges_ranked.iloc[:1].explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 2, "color": "#9999cc"},
                        name="Existing bike network")
viz = edges_ranked.iloc[1:].explore(m=viz, 
                     style_kwds={"weight": 3, "color": "#096a51"},
                        name="Grown bike network")
folium.LayerControl().add_to(viz)
viz

### Relaxing the bike network definition

The import of a custom bike network is useful to change the definition of bike network. For example, the definition could be relaxed to also consider primary streets as protected bike infrastructure, assuming that the city has implemented protected bike lanes along all primary streets:

In [ ]:
custom_filter_relaxed = custom_filter + ['["highway"~"primary"]']

In [ ]:
g = ox.graph_from_place("Municipality of Athens",
                        custom_filter=custom_filter_relaxed,
                        retain_all=True,) #fetch all connected components
ox.io.save_graph_geopackage(g, "Athens_bike_network_relaxed.gpkg")

In [ ]:
edges_ranked_relaxed = gbn.growbikenet("Municipality of Athens",
                               existing_network_spacing='auto',
                               seed_point_linking ='triangulate_delaunay',
                               import_files={"street_network":"Athens_street_network.gpkg",
                                             "bike_network":"Athens_bike_network_relaxed.gpkg"},)

This creates a different looking bicycle network:

In [ ]:
viz = edges_ranked_relaxed.iloc[:1].explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 2, "color": "#9999cc"},
                        name="Existing bike network (relaxed)")
viz = edges_ranked_relaxed.iloc[1:].explore(m=viz, 
                     style_kwds={"weight": 3, "color": "#096a51"},
                        name="Grown bike network")
folium.LayerControl().add_to(viz)
viz

## Import seed points

<a href="usage_01_seed_points.ipynb#File">Usage notebook 1</a> explains how to import custom seed points. 

## Import city boundary

Rest to be written.